In [71]:
# import libary
from datasets import load_dataset

In [72]:
# laod the dataset
dataset = load_dataset("ShenLab/MentalChat16K")

In [73]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 16084
    })
})

In [74]:
# One sample
dataset["train"][0]

{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ",
 'input': "I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.",
 'output': "I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried various approaches like visualization, positive thinking, and medication to manage y

In [75]:
# rename colums with meaningful keys
dataset = dataset["train"].rename_columns({"input":"query","output":"response"})

In [76]:
# One sample
dataset[0]

{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ",
 'query': "I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.",
 'response': "I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried various approaches like visualization, positive thinking, and medication to manage

In [77]:
dataset = dataset.filter(lambda sample: sample["query"] is not None and sample["query"].strip() != "")
dataset = dataset.filter(lambda sample: sample["response"] is not None and sample["response"].strip() != "")


Filter: 100%|██████████| 16057/16057 [00:00<00:00, 64731.49 examples/s]


In [78]:
# compute length
def compute_length(sample, new_key, field_key):
    return {new_key: len(sample[field_key].split())}

In [79]:

dataset = dataset.map(lambda sample: compute_length(sample, new_key="query_length", field_key="query"))
dataset = dataset.map(lambda sample: compute_length(sample, new_key="response_length", field_key="response"))

Map:   9%|▉         | 1432/16057 [00:00<00:01, 14174.39 examples/s]

Map: 100%|██████████| 16057/16057 [00:00<00:00, 18771.49 examples/s]


In [80]:
dataset[0]

{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ",
 'query': "I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.",
 'response': "I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some time now. It's been a challenging journey, and it's commendable that you've tried various approaches like visualization, positive thinking, and medication to manage

In [81]:
from transformers import AutoTokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [82]:
def prepare_dataset_with_text_column(dataset):
   
    
    # Data prompt
    data_prompt = """{}

### Input:
Pateint Query: {}

### Response:
{}"""

    # Formatting the prompt
    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompt(examples):
        sys_prompts = examples["instruction"]
        queries = examples["query"]
        responses = examples["response"]
        texts = []
        
        for sys_prompt, query, response in zip(sys_prompts, queries, responses):
            text = data_prompt.format(sys_prompt, query, response) + EOS_TOKEN
            texts.append(text)
        
        return {"text": texts}
    
    dataset_with_text_column = dataset.map(formatting_prompt, batched=True)
    
    return dataset_with_text_column



In [83]:
dataset = prepare_dataset_with_text_column(dataset)

Map:  87%|████████▋ | 14000/16057 [00:00<00:00, 127317.00 examples/s]

Map: 100%|██████████| 16057/16057 [00:00<00:00, 41416.65 examples/s] 


In [84]:
print(dataset[10]["text"])

You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. 
The assistant gives helpful, comprehensive, and appropriate answers to the user's questions. 

### Input:
Pateint Query: I've been living with my mom for the past five years since she was diagnosed with Alzheimer's. I've taken on the role of her primary caregiver, which has left me feeling drained and emotionally unavailable for my own needs. I've been neglecting my personal relationships and hobbies, and I feel disconnected from my own identity. I've tried to discuss this with my mom's caregivers, but they're so focused on her care that they don't seem to understand or care about my situation. I feel trapped and unable to prioritize my own needs.

### Response:
I can see how challenging this situation is for you, and I appreciate your dedication to your mom's care. It's important to acknowledge the emotional toll this role has taken on you. You've descr

In [85]:
dataset_full = dataset.shuffle(seed=42).train_test_split(train_size=0.8)

In [86]:
dataset_full

DatasetDict({
    train: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 12845
    })
    test: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 3212
    })
})

In [91]:
dataset_train_validation = dataset_full["test"]
dataset_train_validation = dataset_train_validation.train_test_split(test_size=0.5, seed=42)

In [92]:
dataset_train_validation

DatasetDict({
    train: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 1606
    })
    test: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 1606
    })
})

In [93]:
dataset_full["test"] = dataset_train_validation["train"]
dataset_full["validation"] = dataset_train_validation["test"]

In [94]:
dataset_full

DatasetDict({
    train: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 12845
    })
    test: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 1606
    })
    validation: Dataset({
        features: ['instruction', 'query', 'response', 'query_length', 'response_length', 'text'],
        num_rows: 1606
    })
})

In [97]:
# Save to disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset_full.save_to_disk(f"{dataset_based_path}/mh-mental-chat-16k_v1")

Saving the dataset (1/1 shards): 100%|██████████| 1606/1606 [00:00<00:00, 51398.27 examples/s]


In [2]:
from datasets import load_from_disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset/mh-mental-chat-16k_v1"
dataset = load_from_disk(dataset_based_path)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset["train"][5]

{'instruction': "You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions. ",
 'query': '"I\'ve been experiencing extreme levels of stress mainly related to financial difficulties. The constant worry and fear about not being able to make ends meet have led to sleepless nights and increased anxiety. It would be helpful in therapy to explore ways to reduce stress and gain control over my finances."',
 'response': "One of the first steps you can take to reduce stress and gain control over your finances is by creating a budget. Start by listing all your income sources and expenses. This will give you a clear picture of where your money is going and help you identify areas for potential savings. Consider cutting back on non-essential expenses, such as eating out or entertainment, and redirecting that money towards necessiti